# ChestX-ray CAD — train & produce the remaining report figures
This notebook runs the **corrected pipeline** end-to-end on a GPU and generates the six outputs that need a trained image model: **ROC curves, training curves, HPO/model-selection, robustness, bias/fairness, and Grad-CAM**.

**Honest note on scope.** By default it uses the Kaggle **`sample`** subset (~5,600 images, ~4 GB) so it runs on a free Colab GPU in a reasonable time. Numbers from the sample are *real but on a subset* — set `DATASET = 'full'` to reproduce paper-scale results (needs Colab Pro / large disk / hours). Nothing here is fabricated.

**Before you start:** Runtime ▸ Change runtime type ▸ **GPU**.


## 1. Check the GPU


In [ ]:
!nvidia-smi -L


## 2. Install dependencies


In [ ]:
!pip -q install timm 'albumentations>=1.4' opencv-python-headless optuna pyyaml scikit-learn
print('deps installed')


## 3. Get the code
Upload the `chestxray-cad.zip` I gave you (the file picker will appear). Alternatively, if you have pushed the package to GitHub, replace this cell with a `!git clone` of that repo.


In [ ]:
from google.colab import files
print('Upload chestxray-cad.zip ...')
up = files.upload()
import io, zipfile, os, sys
name = next(iter(up))
with zipfile.ZipFile(io.BytesIO(up[name])) as z: z.extractall('/content')

# locate the package root (the folder that holds pyproject.toml + src/chestxray)
ROOT = None
for d, _, fs in os.walk('/content'):
    if 'pyproject.toml' in fs and os.path.isdir(os.path.join(d, 'src', 'chestxray')):
        ROOT = d; break
assert ROOT, 'could not find the chestxray-cad folder after unzip'
os.chdir(ROOT)
SRC = os.path.join(ROOT, 'src')

!pip -q install -e .   # optional; provides console scripts
# Make the package importable NOW without a kernel restart, and for every
# !python subprocess below (they inherit os.environ):
sys.path.insert(0, SRC)
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')

import chestxray
print('OK — chestxray', chestxray.__version__, 'from', chestxray.__file__)
print('cwd:', os.getcwd())


## 4. Download the NIH data from Kaggle
Upload your `kaggle.json` API token (Kaggle ▸ Account ▸ Create New API Token).


In [ ]:
from google.colab import files
print('Upload kaggle.json ...')
files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip -q install kaggle

DATASET = 'sample'   # 'sample' (~4GB, quick) or 'full' (~42GB, paper-scale)
if DATASET == 'sample':
    !kaggle datasets download -d nih-chest-xrays/sample -p /content/nih --unzip
else:
    !kaggle datasets download -d nih-chest-xrays/data -p /content/nih --unzip
print('download complete')


## 5. Arrange the data into the expected layout
The pipeline expects `data/Data_Entry_2017.csv` and images under `data/images*/`.


In [ ]:
import os, sys  # --- ensure chestxray is importable (safe to re-run) ---
if 'ROOT' not in globals() or not os.path.isdir(os.path.join(globals().get('ROOT',''),'src','chestxray')):
    ROOT = next((d for d,_,fs in os.walk('/content') if 'pyproject.toml' in fs and os.path.isdir(os.path.join(d,'src','chestxray'))), None)
assert ROOT, 'chestxray-cad not found — run the upload/unzip cell first'
os.chdir(ROOT); SRC = os.path.join(ROOT,'src')
if SRC not in sys.path: sys.path.insert(0, SRC)
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH','')

import glob, os, shutil
root = os.path.join(ROOT, 'data'); os.makedirs(root, exist_ok=True)

# collect every png under the download into one images folder (symlinks, no copy)
pngs = glob.glob('/content/nih/**/*.png', recursive=True)
img_dir = f'{root}/images_all'; os.makedirs(img_dir, exist_ok=True)
for p in pngs:
    dst = os.path.join(img_dir, os.path.basename(p))
    if not os.path.exists(dst):
        try: os.symlink(p, dst)
        except FileExistsError: pass

# find the labels CSV (full or sample) and expose it under the expected name
cands = (glob.glob('/content/nih/**/Data_Entry_2017.csv', recursive=True)
         or glob.glob('/content/nih/**/sample_labels.csv', recursive=True))
shutil.copy(cands[0], f'{root}/Data_Entry_2017.csv')
print(f'{len(pngs)} images linked; labels: {cands[0]}')


## 6. Configure the run
Small, GPU-friendly settings for the sample. For `DATASET='full'`, raise `min_cases` to 1000 and `epochs` to ~30.


In [ ]:
import os, sys  # --- ensure chestxray is importable (safe to re-run) ---
if 'ROOT' not in globals() or not os.path.isdir(os.path.join(globals().get('ROOT',''),'src','chestxray')):
    ROOT = next((d for d,_,fs in os.walk('/content') if 'pyproject.toml' in fs and os.path.isdir(os.path.join(d,'src','chestxray'))), None)
assert ROOT, 'chestxray-cad not found — run the upload/unzip cell first'
os.chdir(ROOT); SRC = os.path.join(ROOT,'src')
if SRC not in sys.path: sys.path.insert(0, SRC)
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH','')

from chestxray.config import Config
cfg = Config()
cfg.data.data_dir = 'data'; cfg.data.entry_csv = 'Data_Entry_2017.csv'
cfg.data.image_size = 224; cfg.data.num_workers = 2
cfg.data.min_cases = 50 if DATASET == 'sample' else 1000
cfg.model.backbone = 'densenet121'; cfg.model.pretrained = True
cfg.train.loss = 'weighted_bce'; cfg.train.amp = True
cfg.train.epochs = 8 if DATASET == 'sample' else 30
cfg.train.batch_size = 32
cfg.experiment_name = 'colab_run'
cfg.save('configs/colab.yaml')
print(cfg.to_dict())


## 7. Explore the data + sound baselines (real)
Produces the EDA profile and the metadata-only baseline comparison with patient-grouped CV.


In [ ]:
import os, sys  # --- ensure chestxray is importable (safe to re-run) ---
if 'ROOT' not in globals() or not os.path.isdir(os.path.join(globals().get('ROOT',''),'src','chestxray')):
    ROOT = next((d for d,_,fs in os.walk('/content') if 'pyproject.toml' in fs and os.path.isdir(os.path.join(d,'src','chestxray'))), None)
assert ROOT, 'chestxray-cad not found — run the upload/unzip cell first'
os.chdir(ROOT); SRC = os.path.join(ROOT,'src')
if SRC not in sys.path: sys.path.insert(0, SRC)
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH','')

!python scripts/run_eda_baselines.py --csv data/Data_Entry_2017.csv --out artifacts/eda --folds 5


## 8. Train the corrected model
Whole-training-set epochs, class-weighted BCE, patient-disjoint split, early stopping, AMP.


In [ ]:
import os, sys  # --- ensure chestxray is importable (safe to re-run) ---
if 'ROOT' not in globals() or not os.path.isdir(os.path.join(globals().get('ROOT',''),'src','chestxray')):
    ROOT = next((d for d,_,fs in os.walk('/content') if 'pyproject.toml' in fs and os.path.isdir(os.path.join(d,'src','chestxray'))), None)
assert ROOT, 'chestxray-cad not found — run the upload/unzip cell first'
os.chdir(ROOT); SRC = os.path.join(ROOT,'src')
if SRC not in sys.path: sys.path.insert(0, SRC)
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH','')

!python -m chestxray.training.train --config configs/colab.yaml


## 9. Evaluate on the held-out test set
Writes `test_report.json`, `test_predictions.npz` and `test_manifest.csv` (aligned to predictions).


In [ ]:
import os, sys  # --- ensure chestxray is importable (safe to re-run) ---
if 'ROOT' not in globals() or not os.path.isdir(os.path.join(globals().get('ROOT',''),'src','chestxray')):
    ROOT = next((d for d,_,fs in os.walk('/content') if 'pyproject.toml' in fs and os.path.isdir(os.path.join(d,'src','chestxray'))), None)
assert ROOT, 'chestxray-cad not found — run the upload/unzip cell first'
os.chdir(ROOT); SRC = os.path.join(ROOT,'src')
if SRC not in sys.path: sys.path.insert(0, SRC)
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH','')

!python -m chestxray.evaluate --checkpoint artifacts/colab_run/best_model.pt --config configs/colab.yaml


## 10. Results figures — ROC, per-class AUC, training curves


In [ ]:
import os, sys  # --- ensure chestxray is importable (safe to re-run) ---
if 'ROOT' not in globals() or not os.path.isdir(os.path.join(globals().get('ROOT',''),'src','chestxray')):
    ROOT = next((d for d,_,fs in os.walk('/content') if 'pyproject.toml' in fs and os.path.isdir(os.path.join(d,'src','chestxray'))), None)
assert ROOT, 'chestxray-cad not found — run the upload/unzip cell first'
os.chdir(ROOT); SRC = os.path.join(ROOT,'src')
if SRC not in sys.path: sys.path.insert(0, SRC)
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH','')

!python scripts/make_results.py --exp artifacts/colab_run
from IPython.display import Image, display
for f in ['roc_curves.png','per_class_auc.png','training_curves.png']:
    p = f'artifacts/colab_run/{f}'
    if os.path.exists(p): display(Image(p))


## 11. Bias / fairness — macro AUC by subgroup
Joins the test metadata to the predictions and reports AUC per sex / view / age band.


In [ ]:
import os, sys  # --- ensure chestxray is importable (safe to re-run) ---
if 'ROOT' not in globals() or not os.path.isdir(os.path.join(globals().get('ROOT',''),'src','chestxray')):
    ROOT = next((d for d,_,fs in os.walk('/content') if 'pyproject.toml' in fs and os.path.isdir(os.path.join(d,'src','chestxray'))), None)
assert ROOT, 'chestxray-cad not found — run the upload/unzip cell first'
os.chdir(ROOT); SRC = os.path.join(ROOT,'src')
if SRC not in sys.path: sys.path.insert(0, SRC)
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH','')

import numpy as np, pandas as pd, json
import matplotlib.pyplot as plt
from chestxray.evaluation.fairness import fairness_report
from chestxray.data.features import age_band

d = np.load('artifacts/colab_run/test_predictions.npz', allow_pickle=True)
scores, targets, labels = d['scores'], d['targets'], list(d['labels'])
man = pd.read_csv('artifacts/colab_run/test_manifest.csv')
subgroups = {'sex': man['Patient Gender'], 'view': man['View Position'], 'age_band': age_band(man)}
rep = fairness_report(targets, scores, labels, subgroups)
json.dump(rep, open('artifacts/colab_run/fairness.json','w'), indent=2, default=str)
print(json.dumps(rep, indent=2, default=str))

# bar chart per attribute
attrs = [k for k in rep if isinstance(rep[k], dict)]
fig, axes = plt.subplots(1, len(attrs), figsize=(4*len(attrs), 3.5))
if len(attrs) == 1: axes = [axes]
for ax, a in zip(axes, attrs):
    g = rep[a]['per_group']
    ax.bar(range(len(g)), list(g.values()))
    ax.set_xticks(range(len(g))); ax.set_xticklabels(list(g.keys()), rotation=45, fontsize=8)
    ax.axhline(0.5, color='k', ls='--', lw=0.7); ax.set_ylim(0.5, 1.0)
    ax.set_title(f'{a} (gap {rep[a]["auc_gap"]:.3f})'); ax.set_ylabel('macro AUC')
plt.tight_layout(); plt.savefig('artifacts/colab_run/fairness.png', dpi=150); plt.show()


## 12. Robustness — AUC under image corruptions
Measures macro-AUC degradation under noise, blur, brightness and rotation on a test subset.


In [ ]:
import os, sys  # --- ensure chestxray is importable (safe to re-run) ---
if 'ROOT' not in globals() or not os.path.isdir(os.path.join(globals().get('ROOT',''),'src','chestxray')):
    ROOT = next((d for d,_,fs in os.walk('/content') if 'pyproject.toml' in fs and os.path.isdir(os.path.join(d,'src','chestxray'))), None)
assert ROOT, 'chestxray-cad not found — run the upload/unzip cell first'
os.chdir(ROOT); SRC = os.path.join(ROOT,'src')
if SRC not in sys.path: sys.path.insert(0, SRC)
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH','')

import cv2, numpy as np, pandas as pd, json
import matplotlib.pyplot as plt
from chestxray.inference import Predictor
from chestxray.evaluation.robustness import robustness_report

pred = Predictor('artifacts/colab_run/best_model.pt', image_size=cfg.data.image_size)
man = pd.read_csv('artifacts/colab_run/test_manifest.csv').head(300).reset_index(drop=True)
imgs = [cv2.imread(p, cv2.IMREAD_GRAYSCALE) for p in man['path']]
yt = man[labels].to_numpy(float)

def predict_fn(images):
    return np.stack([np.array(list(pred.predict(im).values())) for im in images])

rob = robustness_report(imgs, yt, labels, predict_fn)
json.dump(rob, open('artifacts/colab_run/robustness.json','w'), indent=2)
print(json.dumps(rob, indent=2))

kinds = list(rob['corruptions']); aucs = [rob['corruptions'][k]['auc'] for k in kinds]
plt.figure(figsize=(6,4))
plt.axhline(rob['clean_auc'], color='g', ls='--', label=f"clean {rob['clean_auc']:.3f}")
plt.bar(range(len(kinds)), aucs, color='#C44E52')
plt.xticks(range(len(kinds)), kinds, rotation=30); plt.ylabel('macro AUC')
plt.title('Robustness to corruptions'); plt.legend()
plt.tight_layout(); plt.savefig('artifacts/colab_run/robustness.png', dpi=150); plt.show()


## 13. Explainability — Grad-CAM
Heat-map over the regions driving a prediction; check it attends to the lung field, not markers/borders.


In [ ]:
import os, sys  # --- ensure chestxray is importable (safe to re-run) ---
if 'ROOT' not in globals() or not os.path.isdir(os.path.join(globals().get('ROOT',''),'src','chestxray')):
    ROOT = next((d for d,_,fs in os.walk('/content') if 'pyproject.toml' in fs and os.path.isdir(os.path.join(d,'src','chestxray'))), None)
assert ROOT, 'chestxray-cad not found — run the upload/unzip cell first'
os.chdir(ROOT); SRC = os.path.join(ROOT,'src')
if SRC not in sys.path: sys.path.insert(0, SRC)
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH','')

import torch, cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from chestxray.models import build_model
from chestxray.explain import GradCAM, default_target_layer, overlay
from chestxray.data.transforms import build_transforms

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load('artifacts/colab_run/best_model.pt', map_location=dev)
model = build_model(ckpt['config']['model']['backbone'], len(labels), pretrained=False).to(dev)
model.load_state_dict(ckpt['model_state']); model.eval()
cam = GradCAM(model, default_target_layer(model))
tf = build_transforms(cfg.data.image_size, train=False)

man = pd.read_csv('artifacts/colab_run/test_manifest.csv')
target = 'Cardiomegaly' if 'Cardiomegaly' in labels else labels[0]
cls = labels.index(target)
row = man[man[target] == 1].iloc[0]
img = cv2.imread(row['path'], cv2.IMREAD_GRAYSCALE); img3 = np.stack([img]*3, -1)
x = tf(image=img3)['image'].unsqueeze(0).to(dev)
hm = cam(x, cls)
ov = overlay(hm, img)
cv2.imwrite('artifacts/colab_run/gradcam.png', ov)
plt.figure(figsize=(5,5)); plt.imshow(cv2.cvtColor(ov, cv2.COLOR_BGR2RGB))
plt.title(f'Grad-CAM: {target}'); plt.axis('off'); plt.show()


## 14. (Optional) Hyperparameter optimisation / model selection
Each trial trains a full model, so this is slow. Keep `--trials` small on the sample.


In [ ]:
import os, sys  # --- ensure chestxray is importable (safe to re-run) ---
if 'ROOT' not in globals() or not os.path.isdir(os.path.join(globals().get('ROOT',''),'src','chestxray')):
    ROOT = next((d for d,_,fs in os.walk('/content') if 'pyproject.toml' in fs and os.path.isdir(os.path.join(d,'src','chestxray'))), None)
assert ROOT, 'chestxray-cad not found — run the upload/unzip cell first'
os.chdir(ROOT); SRC = os.path.join(ROOT,'src')
if SRC not in sys.path: sys.path.insert(0, SRC)
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH','')

# Uncomment to run (expensive):
# !python -m chestxray.training.hpo --config configs/colab.yaml --trials 4 --out artifacts/hpo_result.json
# import json; print(json.load(open('artifacts/hpo_result.json')))


## 15. Bundle the results and download
Send `results_bundle.zip` back and I will slot the real figures and numbers into the paper.


In [ ]:
import os, sys  # --- ensure chestxray is importable (safe to re-run) ---
if 'ROOT' not in globals() or not os.path.isdir(os.path.join(globals().get('ROOT',''),'src','chestxray')):
    ROOT = next((d for d,_,fs in os.walk('/content') if 'pyproject.toml' in fs and os.path.isdir(os.path.join(d,'src','chestxray'))), None)
assert ROOT, 'chestxray-cad not found — run the upload/unzip cell first'
os.chdir(ROOT); SRC = os.path.join(ROOT,'src')
if SRC not in sys.path: sys.path.insert(0, SRC)
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH','')

!cd artifacts && zip -qr /content/results_bundle.zip colab_run eda
from google.colab import files
files.download('/content/results_bundle.zip')


---
**What this produced** (in `artifacts/colab_run/`): `test_report.json`, `results_perclass.csv`, `roc_curves.png`, `per_class_auc.png`, `training_curves.png`, `fairness.json`+`.png`, `robustness.json`+`.png`, `gradcam.png` — plus EDA/baselines in `artifacts/eda/`. All computed on real data.
